# SliceWeave: independent Kaggle M0–M4c study

This notebook is self-contained: it has no dependency on the SliceWeave repository, no local model cache requirement, and downloads Qwen models directly from Hugging Face. It evaluates **Qwen3-4B, Qwen3-8B, and Qwen3-14B sequentially**, releasing memory after every model.

The protocol is deliberately full-scale: every canonical SliceWeave prompt plus 120 deterministic arithmetic draws is used (594 unique prompts after exact duplicates are removed). Artifacts are written to `/kaggle/working/SliceWeaveResults/<model>/results/<stage>/{csv,json,plots,logs,traces}`. Set `SELECTED_MODELS` to one entry only when resuming a failed or interrupted model; this is not a pilot setting.


In [1]:
# Kaggle already includes most packages.  This makes a fresh Kaggle image reproducible.
import sys, subprocess, importlib.util
required = {'transformers': 'transformers>=4.51', 'accelerate': 'accelerate>=1.5',
            'huggingface_hub': 'huggingface_hub>=0.30', 'matplotlib': 'matplotlib>=3.8'}
missing = [spec for name, spec in required.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('dependency check complete')


dependency check complete


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import csv, gc, hashlib, json, os, platform, random, time
from collections import Counter

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

OUTPUT_ROOT = Path('/kaggle/working/SliceWeaveResults')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODELS = {
    'qwen3_4b': 'Qwen/Qwen3-4B',
    'qwen3_8b': 'Qwen/Qwen3-8B',
    'qwen3_14b': 'Qwen/Qwen3-14B',
}
SELECTED_MODELS = ['qwen3_4b', 'qwen3_8b', 'qwen3_14b']

# Full protocol settings.  Do not lower these for a reported study.
GROUP_SIZE = 128
# 498 canonical prompts + 120 deterministic variations = 618 full-study prompts.
ARITHMETIC_PROMPTS = 120
M1_LAYERS, ROUTER_LAYERS = 5, 3
M1_POSITIONS, M4C_POSITIONS = 12, 4
PROBE_EPOCHS = 160
OVERWRITE_CAPTURE = False
RETAIN = (1.0, .70, .50, .40, .30, .20, .15, .10, .05)
M4C_RETAIN = (.05, .10, .15, .20, .30, .50)

print({'output_root': str(OUTPUT_ROOT), 'models': SELECTED_MODELS, 'protocol': 'full'})


{'output_root': '\\kaggle\\working\\SliceWeaveResults', 'models': ['qwen3_4b', 'qwen3_8b', 'qwen3_14b'], 'protocol': 'full'}


## Canonical prompt corpus

Each category is deliberately presented in its own complete block. These are the exact 498 curated prompts used by the local SliceWeave protocol; they are embedded here so Kaggle needs no repository checkout.

In [3]:
# Initialize the embedded canonical corpus.
CATEGORIES = {}


### `coding`

In [4]:
# coding: 201 canonical prompts
CATEGORIES['coding'] = [
    "Write a Python function to reverse a linked list in place.",
    "Implement an LRU cache in Python with O(1) get and put.",
    "Implement a thread-safe bounded queue in Python using condition variables.",
    "Why does mutating a list while iterating over it skip elements? Show the fix.",
    "How do I make an HTTP call retry with exponential backoff in Python?",
    "What's the difference between __new__ and __init__ in Python, with a singleton example?",
    "Explain Python's GIL: when do threads help and when do processes win?",
    "Write a generator that yields lines from a 10 GB log file without loading it.",
    "How do I profile a slow Python function with cProfile and read the output?",
    "Convert this class-based code to use dataclasses with validation.",
    "Write a context manager that times a block and logs slow executions above 100ms.",
    "How do I safely share a numpy array between multiprocessing workers?",
    "Implement a retry decorator that swallows only transient HTTP errors.",
    "What is the fastest way to dedupe a list of 10M dicts by key in Python?",
    "Why does my asyncio task never run? Explain event loop starvation.",
    "Write a script that watches a directory and processes new CSV files idempotently.",
    "Explain pickling pitfalls with lambdas and nested functions.",
    "Write a rate limiter using a token bucket, thread-safe, with tests.",
    "My pandas merge exploded memory. Walk through diagnosing dtype blowup.",
    "Why does sys.setrecursionlimit not save my deep recursion? Alternatives?",
    "Write a C function that reverses a string in place without extra memory.",
    "Why does my C program segfault when freeing a pointer twice?",
    "Explain the difference between malloc, calloc, and realloc failure modes.",
    "Write a C function to detect a cycle in a singly linked list.",
    "How do dangling pointers arise after realloc? Show a safe pattern.",
    "Implement a bounded strcpy-style copy and explain the overflow risk.",
    "What is strict aliasing and when does it break type-punning code?",
    "Explain why returning a pointer to a stack local is undefined behavior.",
    "Write a bit-twiddling function to count set bits and explain the trick.",
    "How does alignment affect casting a char buffer to a struct pointer?",
    "Implement a fixed-size ring buffer in C with head/tail indices.",
    "Explain volatile: when is it needed and when is it not a synchronization tool?",
    "Write signal-safe code: what can you do inside a signal handler?",
    "How do I debug a heap corruption that crashes far from the corrupting write?",
    "What is the difference between undefined, unspecified, and implementation-defined?",
    "Explain endianness pitfalls when serializing structs over a network.",
    "How do I use valgrind to find an off-by-one heap read?",
    "Why can sizeof('a') equal 4 in C but 1 in C++?",
    "Write a macro-free min/max for mixed integer types and explain pitfalls.",
    "Explain what happens when a buffer overflow overwrites a return address.",
    "Why does my C++ program segfault when freeing a pointer twice?",
    "Explain move semantics: what does std::move actually do, and not do?",
    "When should I use std::unique_ptr versus std::shared_ptr, with examples?",
    "Why is returning const std::string& from a function dangerous after C++11?",
    "Implement a scoped lock wrapper using RAII for a legacy mutex API.",
    "What is the rule of five and when can I default everything?",
    "Explain vector reallocation and iterator invalidation with a crash example.",
    "How does std::optional improve APIs over sentinel values?",
    "Why can template instantiation explode compile times? Mitigations?",
    "Explain SFINAE versus concepts with a before/after example.",
    "What is a vtable and when does virtual dispatch not happen?",
    "Implement a small thread pool in C++17 with futures for results.",
    "Why is std::vector<bool> weird, and what should I use instead?",
    "Explain dangling references with std::string_view and lifetime rules.",
    "Explain undefined behavior with signed overflow and optimizer assumptions.",
    "When is emplace_back faster than push_back, and when is it identical?",
    "Implement a type-safe variant visitor using std::visit and overload sets.",
    "Why does my lambda capture dangle when stored in a callback registry?",
    "Explain memory ordering for std::atomic with acquire/release in a spinlock.",
    "How do I write a custom allocator-aware container, minimal version?",
    "Convert this recursive Fibonacci to a memoized iterative version in Java.",
    "Explain ConcurrentHashMap vs synchronized HashMap for a read-heavy cache.",
    "Implement a Producer-Consumer with BlockingQueue and graceful shutdown.",
    "What causes a Java OutOfMemoryError: metaspace, and how do I diagnose it?",
    "Explain equals/hashCode contracts; what breaks a HashSet lookup?",
    "Write a stream pipeline that groups orders by month and sums revenue.",
    "Why is double-checked locking broken without volatile?",
    "Explain GC pause types in the JVM and how G1 mitigates them.",
    "How do virtual threads change blocking-IO server design in Java 21?",
    "Implement a custom AutoCloseable resource with correct exception handling.",
    "Why are records better than Lombok data classes for DTOs?",
    "Explain classloader leaks in application servers with static caches.",
    "What is the difference between checked and unchecked exceptions by design intent?",
    "Implement a bounded rate limiter with Semaphore and a scheduled refill.",
    "Why can floating point money calculations fail? Show BigDecimal usage.",
    "Explain immutability benefits with a thread-safe value object example.",
    "How do I profile a Java service to find lock contention hotspots?",
    "Explain the module system's strong encapsulation vs classpath JARs.",
    "What does String.intern do and when does it help parsing hot paths?",
    "Why can an unbounded thread pool exhaust memory under burst load?",
    "Implement Dijkstra in Rust using a binary heap.",
    "What does the Rust borrow checker complain about when I mutate a Vec while iterating?",
    "Explain ownership move semantics when passing a String into a function.",
    "When do I choose Rc<RefCell<T>> versus Arc<Mutex<T>>?",
    "Write a Rust iterator adapter chain that parses CSV rows into structs.",
    "What is a lifetime 'a and how do I annotate a struct holding a reference?",
    "Why does the compiler reject returning a reference to a local? Explain elision.",
    "Implement a trait with default methods and explain dynamic vs static dispatch.",
    "Explain Result propagation with the ? operator and custom error types.",
    "What does Send + Sync mean for spawning threads with std::thread?",
    "How do I safely share mutable state across await points in async Rust?",
    "Explain Box, Rc, Arc, and when each is the right tool.",
    "Write a Rust function using pattern matching on an enum state machine.",
    "Why does closure capture move or borrow depending on usage?",
    "Implement a linked list in Rust and discuss why it is famously hard.",
    "Explain unsafe blocks: what responsibilities does the author take on?",
    "What is zero-cost abstraction with an iterator optimization example?",
    "Explain panics versus Result for recoverable versus unrecoverable errors.",
    "How does Cargo handle feature flags and conditional compilation?",
    "What does the borrow checker guarantee about data races in safe Rust?",
    "Explain Go goroutine leaks from unbuffered channels and how to fix them.",
    "Why does Go prefer composition over inheritance? Show embedding.",
    "Implement a worker pool in Go with graceful shutdown via context.",
    "What is the difference between make and new in Go?",
    "Explain Go's panic/recover philosophy versus exceptions.",
    "How does the sync.WaitGroup work and what does misuse look like?",
    "Write a concurrent web scraper in Go with a bounded semaphore channel.",
    "Why are Go interfaces satisfied implicitly, and what does that enable?",
    "Explain the nil interface gotcha with a typed nil pointer.",
    "How does Go's scheduler multiplex goroutines onto OS threads?",
    "Implement a rate-limited HTTP client using time.Ticker and context.",
    "Explain slices versus arrays: what does append do under the hood?",
    "How do I write table-driven tests in Go with subtests?",
    "Explain defer ordering and evaluation of arguments at defer time.",
    "Implement a JSON API handler with proper error wrapping in Go.",
    "Explain context cancellation propagation through a call tree.",
    "How do modules, versioning, and the go.sum file work?",
    "What does go vet catch that compilers do not? Examples.",
    "Why is sharing memory by communicating preferred over mutexes here?",
    "What is the zero value and why does Go avoid uninitialized variables?",
    "Refactor this callback pyramid into async/await JavaScript.",
    "Why is my React useEffect firing twice in development?",
    "Explain the event loop: microtasks versus macrotasks with Promise ordering.",
    "What does 'this' refer to in arrow versus regular functions?",
    "Write a TypeScript type that extracts the return type of a generic function.",
    "Explain closure-based memoization and its memory tradeoffs in JS.",
    "Implement a debounce util in TypeScript with cancel and flush.",
    "Explain Promise.all vs allSettled vs any with failure scenarios.",
    "How do I avoid prop drilling with context without re-render storms?",
    "Write a typed event emitter in TypeScript with on/off/emit.",
    "Why can mutation of state break memoized components? Immutability pattern.",
    "Implement retries with exponential backoff for fetch in the browser.",
    "Explain CORS preflight: what triggers OPTIONS and how to fix failures.",
    "Write a Node.js stream pipeline that transforms a large CSV safely.",
    "Explain the module systems: ESM vs CommonJS interop pitfalls.",
    "How does garbage collection handle detached DOM nodes causing leaks?",
    "Implement a typed API client wrapper with discriminated union results.",
    "Explain structural typing in TypeScript with a surprising acceptance example.",
    "Why does typeof null return 'object'? Practical guard patterns.",
    "Explain shallow vs deep reactivity in frontend state libraries.",
    "Explain the difference between a SQL inner join and left join.",
    "Write a PostgreSQL query to find duplicate emails using a window function.",
    "Write a SQL migration to add a nullable column with a backfill strategy.",
    "Explain when a composite index helps and when it does nothing.",
    "Write a recursive CTE to flatten an org chart with cycle protection.",
    "Why does OR sometimes prevent index usage? Rewrite with UNION.",
    "Explain isolation levels: what anomaly does READ COMMITTED still allow?",
    "Write a query to find the top 3 products per category using row_number.",
    "How do I safely change a column type on a large table with zero downtime?",
    "Explain N+1 queries in an ORM and fix with eager loading.",
    "Write an upsert that handles concurrent inserts without deadlocks.",
    "Explain MVCC and how vacuum reclaims space in PostgreSQL.",
    "Write a query detecting gaps in a sequence of daily events.",
    "Explain EXPLAIN plans: what makes a seq scan acceptable?",
    "Write a query to pivot monthly revenue into columns.",
    "How do partial indexes speed up soft-delete filtered queries?",
    "Explain locking from FK checks and how to avoid deadlocks on delete.",
    "Why is SELECT * bad for wide tables in production code?",
    "Explain the tradeoffs of denormalization for a read-heavy dashboard.",
    "What does ON DELETE CASCADE do and when is it dangerous?",
    "Write a bash one-liner to find the 10 largest files under /var recursively.",
    "Explain set -euo pipefail and what breaks without it.",
    "Write a script that retries a flaky curl with backoff and logs failures.",
    "How do I safely handle filenames with spaces in a for loop?",
    "Explain process substitution: diff two command outputs directly.",
    "Write an awk one-liner to sum the second column grouped by the first.",
    "What is the difference between $@ and $* inside quotes?",
    "Write a trap handler that cleans temp files on SIGINT.",
    "How do I atomically replace a config file from a deploy script?",
    "Explain exit codes and how to make a script pipeline-fail on inner failure.",
    "Write a jq pipeline to extract and count error types from JSON logs.",
    "Explain xargs quoting pitfalls with -I and spaces.",
    "Write a portable shell function to check if a TCP port is open.",
    "Explain cron vs systemd timers for a job needing env secrets.",
    "Write a script to diff two directory trees by content hash.",
    "How do I tail a log and alert on a regex within the last N lines?",
    "Write a systemd unit with restart policy and a health check timer.",
    "How do I limit a command's memory with ulimit and verify it?",
    "Explain what happens when a shell variable is referenced while unset.",
    "Write a here-document that safely embeds a SQL script with variables.",
    "How do I speed up a slow find command over millions of files?",
    "Implement binary search on a rotated sorted array; state the invariants.",
    "Implement an LRU cache in O(1) using a hashmap plus doubly linked list.",
    "Explain union-find with path compression and rank, then solve accounts merge.",
    "Write topological sort with cycle detection for a build system.",
    "Implement a segment tree for range sum with point update.",
    "Explain Dijkstra versus Bellman-Ford and when negative edges matter.",
    "Write a sliding window solution for longest substring without repeats.",
    "Implement a trie for autocomplete with prefix counts.",
    "Explain dynamic programming for edit distance with the recurrence.",
    "Write KMP failure function and explain why naive search is O(nm).",
    "Implement quickselect for median and analyze expected time.",
    "Explain monotonic stack with the next-greater-element problem.",
    "Implement Floyd's cycle detection and prove the meeting point math.",
    "Write a heap-based solution for merging k sorted lists.",
    "Implement Kruskal's MST with a disjoint set and compare with Prim's.",
    "Explain the two-pointer technique for pair-sum in a sorted array.",
    "Write a matrix rotation in place for square matrices.",
    "Implement interval merging after sorting, with edge cases.",
    "Explain hashing collisions and open addressing load factor tradeoffs.",
    "Write a bitset solution for subset-sum with a memory analysis.",
]


### `debugging`

In [5]:
# debugging: 20 canonical prompts
CATEGORIES['debugging'] = [
    "Debug this FastAPI JWT authentication bug: the token expires before refresh.",
    "Debug why my Docker container can't resolve DNS inside the build stage.",
    "Why does float arithmetic make 0.1 + 0.2 != 0.3 and how do languages mitigate it?",
    "My service works locally but 503s in staging with 4x traffic. Systematic debug plan?",
    "Why does my test pass alone but fail in the full suite? Ordering or state?",
    "Intermittent timeout at p99 only. What do you check first and why?",
    "My regex matches greedily across lines. Explain and fix lazily.",
    "A memory leak appears only under HTTPS. What instrumentation do you add?",
    "Why does my retry make the outage worse? Explain retry storms and jitter.",
    "The API returns stale data after deploy. Which caches invalidate and when?",
    "Bug reproduces only on ARM. What classes of bugs behave differently there?",
    "My binary works with -O0 but crashes with -O2. Likely culprits and how to bisect?",
    "Explain reading a flame graph: where do I start with a 3x regression?",
    "My docker build broke after a base image update. How do you pin and diff?",
    "Why do two identical requests return different orderings? Non-determinism sources.",
    "Kafka consumer lags then recovers nightly. What does that pattern indicate?",
    "A query jumped from 5ms to 4s after an insert-heavy day. Diagnose step by step.",
    "Why does my UI flash unstyled content only on cold start?",
    "Production-only NullPointerException: what must logs carry to debug async flows?",
    "My cron ran twice at a DST boundary. How do schedules handle time ambiguity?",
]


### `concurrency`

In [6]:
# concurrency: 20 canonical prompts
CATEGORIES['concurrency'] = [
    "Explain memory ordering: why can a lock-free counter miss increments?",
    "Explain a deadlock with lock ordering and fix a transfer-money example.",
    "What is a data race versus a race condition, with a Java and a C++ example?",
    "Explain compare-and-swap and implement a lock-free stack sketch.",
    "Why does double-checked locking need volatile in Java or atomics in C++?",
    "Explain goroutine leaks with a sender that never closes its channel.",
    "What is the thundering herd on a broadcast wake and how do you tame it?",
    "Implement a reader-writer lock policy for a hot config cache. Pitfalls?",
    "Explain ABA in lock-free algorithms and a tagged-pointer mitigation.",
    "Why can two threads see different sizes of the same concurrent map?",
    "Explain false sharing on cache lines with a counter-array benchmark.",
    "What does 'happens-before' mean and how do synchronization edges establish it?",
    "Design a bounded pipeline with backpressure across 3 stages.",
    "Why are idempotency keys needed for at-least-once queues? Show a duplicate.",
    "Explain cancellation: cooperative checks vs thread interruption semantics.",
    "How would you test for races systematically? Sanitizers, stress tests, model checkers.",
    "Explain optimistic vs pessimistic concurrency for a booking seat flow.",
    "Why does one slow consumer stall a shared queue? Fairness designs.",
    "Implement a single-flight guard that prevents cache stampede on miss.",
    "What goes wrong if a signal handler allocates memory? Async-signal-safety.",
]


### `security`

In [7]:
# security: 20 canonical prompts
CATEGORIES['security'] = [
    "Explain SQL injection with a vulnerable query and the parameterized fix.",
    "Why is storing passwords with SHA-256 alone wrong? Describe bcrypt/argon2 use.",
    "Explain CSRF and how SameSite plus tokens mitigate it.",
    "What is a JWT's weakness when it cannot be revoked? Refresh token design.",
    "Explain XSS types and a Content-Security-Policy that blocks inline scripts.",
    "Why is client-side validation never sufficient? Server-side trust boundaries.",
    "Explain a timing attack on string comparison and constant-time equality.",
    "What does TLS 1.3 change versus TLS 1.2 in handshake security?",
    "Explain IDOR with an API example and an object-level authz fix.",
    "Why should secrets never live in env files committed to git? Rotation plan.",
    "Explain SSRF with a webhook URL feature and egress filtering mitigation.",
    "What is a deserialization gadget chain and why is it dangerous?",
    "Explain least privilege with a CI service account example.",
    "How does a nonce prevent replay in authentication protocols?",
    "Explain secure random versus math random with a token generation bug.",
    "What is clickjacking and which headers prevent framing?",
    "Explain rate limiting and lockout tradeoffs in login endpoints.",
    "Why is logging sensitive data a compliance problem? Redaction strategies.",
    "Explain dependency confusion with a private package name example.",
    "What is privilege escalation via mass assignment and the fix in a serializer?",
]


### `testing`

In [8]:
# testing: 20 canonical prompts
CATEGORIES['testing'] = [
    "Design a test pyramid for a REST API: what lives at each layer?",
    "Write unit tests for a rate limiter including boundary and concurrency cases.",
    "What makes a test flaky and how do you quarantine and fix flakiness?",
    "Explain AAA structure and why one assertion per concept improves failures.",
    "How do you test async code deterministically? Fake clocks and schedulers.",
    "Write property-based tests for a JSON parser with shrinking examples.",
    "Explain test doubles: dummy, fake, stub, spy, mock with a checkout example.",
    "What is contract testing between microservices and what does it catch?",
    "How do you test migration scripts against production-like data safely?",
    "Write a golden-master test for a report generator refactoring.",
    "Explain coverage's limits: what does 100% coverage fail to catch?",
    "How do you fuzz an HTTP API for 500s and unhandled input classes?",
    "Design integration tests with Testcontainers for a Postgres-backed service.",
    "Explain snapshot testing pitfalls for UI components and when it helps.",
    "What is mutation testing and what does it reveal about assertion strength?",
    "How do you load test to find the knee, not just the max, of throughput?",
    "Write E2E tests that survive UI refactors: selector strategy.",
    "Explain chaos testing for a multi-AZ service: first three experiments.",
    "How do you test timezones in a scheduling feature across DST boundaries?",
    "What is a test fixture lifecycle per suite vs per test and why it matters.",
]


### `performance`

In [9]:
# performance: 20 canonical prompts
CATEGORIES['performance'] = [
    "A p99 jumped 10x after adding auth middleware. Profile plan for a Node service.",
    "Explain B-tree page cache behavior when an index outgrows RAM.",
    "Why is an O(n log n) sort slower than an O(n^2) one on 50 items? Constant factors.",
    "Explain branch prediction with a sorted-vs-unsorted loop benchmark.",
    "How does batch size affect throughput vs latency in a writer? Model the tradeoff.",
    "Explain why sequential IO beats random IO on NVMe and how to exploit it.",
    "A cache hit ratio is 99% but latency is high. What else dominates?",
    "Explain gzip level tradeoffs for a 10x traffic surge on static assets.",
    "How do you decide between adding indexes versus denormalizing a summary table?",
    "Explain connection pool sizing math for 50 rps and 40ms queries.",
    "Why does an ORM lazy-load cause 1000 queries on a list page? Fix patterns.",
    "What is tail latency amplification across 5 dependent services?",
    "Explain JIT warmup for a Java batch job and how to keep profiles warm.",
    "How would you reduce cold start time of a serverless function 5x?",
    "Explain the memory-vs-speed tradeoff of bloom filters in a key-value store.",
    "Why is per-request JSON parsing CPU-heavy at 10k rps? Alternatives.",
    "Explain NUMA effects for a memory-bound multithreaded benchmark.",
    "What makes a query parallelize poorly? Skew and barrier examples.",
    "Explain allocation churn in Go and GC tuning with pprof profiles.",
    "How do you benchmark honestly: warmup, variance, pinning, and reporting.",
]


### `code_review`

In [10]:
# code_review: 20 canonical prompts
CATEGORIES['code_review'] = [
    "Review this diff adding a global mutable config: what are the risks and fixes?",
    "A PR swallows exceptions with bare except. What feedback do you leave and why?",
    "Review a PR that introduces an N+1 ORM query behind a loop. Suggest fixes.",
    "How do you review for security in an auth PR? Checklist with concrete checks.",
    "A teammate's PR works but has no tests. Constructive review comment framework.",
    "Review a retry loop without backoff or jitter in a client library.",
    "What do you look for in a migration PR that alters a hot table?",
    "Review exposing an internal endpoint without authz. Severity and remediation.",
    "A PR mixes refactoring with feature changes. How do you ask to split and why?",
    "Review inconsistent error handling across a new module. Suggest a policy.",
    "How do you flag premature abstraction kindly, with a concrete simpler design?",
    "Review a PR adding a dependency: what supply-chain questions do you ask?",
    "A PR renames 400 files. How do you review semantics vs noise effectively?",
    "Review a caching PR for invalidation correctness and stampede risk.",
    "What comments belong in code vs docs vs issue tracker? With examples.",
    "Review a public API change for backward compatibility and versioning strategy.",
    "How do you review logging changes for PII and volume cost?",
    "A PR adds a benchmark. What makes benchmark code trustworthy in review?",
    "Review concurrency additions: locks, atomics, and task lifetime checks.",
    "Give feedback style rules: small nits vs blockers, tone, and blocking criteria.",
]


### `api_system_design`

In [11]:
# api_system_design: 20 canonical prompts
CATEGORIES['api_system_design'] = [
    "Design a URL shortener: id strategy, redirects, analytics, and cache layout.",
    "Design a rate limiter for a multi-tenant API: algorithms and storage.",
    "Design a notification service: fanout, dedupe, and delivery guarantees.",
    "Design pagination for an infinite feed: cursor scheme and stability.",
    "Design a webhook delivery system with retries, signatures, and ordering.",
    "Design an image resizing pipeline: async jobs, CDN, and cost control.",
    "Design a feature flag system: evaluation, targeting, and consistency.",
    "Design idempotent payment APIs: keys, retries, and reconciliation.",
    "Design a search autosuggest service: indexing and freshness budget.",
    "Design a multi-region read model: replication lag and conflict policy.",
    "Design a file upload service: direct-to-storage, presigning, virus scan.",
    "Design an audit log: immutability, retention, and query patterns.",
    "Design a job scheduler: cron semantics, sharding, and missed-run policy.",
    "Design a real-time collaboration service: OT vs CRDT tradeoffs.",
    "Design a rate-limited public API SDK: retries, backoff, and circuit breaking.",
    "Design a metrics pipeline: cardinality control and downsampling.",
    "Design a session store: cookie vs server-side and revocation needs.",
    "Design a config service: versioning, rollback, and client caching.",
    "Design an offline-first mobile sync: conflict resolution strategies.",
    "Design an email ingestion pipeline: parsing, spam scoring, attachments.",
]


### `devops_cloud`

In [12]:
# devops_cloud: 20 canonical prompts
CATEGORIES['devops_cloud'] = [
    "Debug why my Kubernetes pod is CrashLoopBackOff: ordered triage steps.",
    "Explain blue-green versus canary deployments and rollback triggers.",
    "Write a GitHub Actions workflow that caches dependencies and runs matrix tests.",
    "What belongs in a Dockerfile for a Python service to stay small and cacheable?",
    "Explain liveness vs readiness probes with a slow-init Java app.",
    "How do you set SLOs and error budgets for a checkout API?",
    "Explain structured logging with correlation IDs across async boundaries.",
    "Design a database migration rollback plan for a 3-step schema change.",
    "What is a postmortem template that avoids blame and yields actions?",
    "Explain autoscaling signals: CPU vs queue depth vs latency-based scaling.",
    "How do you safely rotate database credentials with zero downtime?",
    "Explain IaC drift: detection and reconciliation in Terraform.",
    "What is a runbook and what makes one usable during an incident?",
    "Explain TLS termination choices at LB vs app with cert rotation.",
    "How do you capacity plan a cache: working set estimation and eviction checks?",
    "Explain graceful shutdown: signal handling and connection draining.",
    "What is cost observability: tagging, unit economics, and alerts on spend?",
    "Explain feature flags in a deploy pipeline: kill switches and cleanup.",
    "How do you version APIs deployed independently across teams?",
    "Explain incident severity levels and on-call escalation design.",
]


### `math`

In [13]:
# math: 25 canonical prompts
CATEGORIES['math'] = [
    "Solve: x^2 - 5x + 6 = 0.",
    "Prove that the square root of 2 is irrational.",
    "Compute the derivative of sin(x)*e^x.",
    "What is the sum of the first 100 positive integers?",
    "Explain dynamic programming using the coin change problem.",
    "Find the eigenvalues of the matrix [[2,1],[1,2]].",
    "Integrate x^2 * ln(x) dx by parts.",
    "What is the probability of getting exactly 3 heads in 5 fair coin flips?",
    "Prove by induction that 1+2+...+n = n(n+1)/2.",
    "Solve the recurrence T(n) = 2T(n/2) + n using the master theorem.",
    "What is the determinant of a 3x3 matrix with rows (1,2,3),(4,5,6),(7,8,10)?",
    "Find the equation of the tangent line to y = x^3 at x = 2.",
    "How many ways can you arrange the letters of MISSISSIPPI?",
    "Explain why e^(i*pi) + 1 = 0.",
    "Compute the limit of (1 + 1/n)^n as n goes to infinity.",
    "A bag has 4 red and 6 blue balls; draw 2 without replacement. P(both red)?",
    "Find the area under y = sin(x) from 0 to pi.",
    "What is 17 * 23 using the difference of squares trick?",
    "Prove that there are infinitely many prime numbers.",
    "Solve the system: 2x + 3y = 12, x - y = 1.",
    "What is the expected value of rolling two six-sided dice?",
    "Explain the difference between a martingale and a random walk.",
    "Explain Bayes' theorem with a medical screening example.",
    "What is the pigeonhole principle? Give two nontrivial applications.",
    "Solve |x - 3| + |x + 1| = 6 and explain the case split.",
]


### `reasoning`

In [14]:
# reasoning: 24 canonical prompts
CATEGORIES['reasoning'] = [
    "If all bloops are razzies and all razzies are lazzies, are all bloops lazzies?",
    "A farmer has 17 sheep and all but 9 run away. How many remain?",
    "Explain why a red herring fallacy weakens an argument.",
    "Order these: losing keys, finding keys, locking door, leaving house.",
    "Which is heavier: a kilogram of steel or two pounds of feathers?",
    "Alice is faster than Bob, Bob is faster than Carol. Is Alice slower than Carol?",
    "If it rains, the street gets wet. The street is wet. Did it necessarily rain?",
    "A bat and a ball cost 1.10 together; the bat costs 1.00 more than the ball. Ball price?",
    "Explain the difference between correlation and causation with an example.",
    "Five people shake hands once with each other. How many handshakes?",
    "You have a 3L and a 5L jug; measure exactly 4 liters.",
    "Is the statement 'this sentence is false' resolvable? Explain the paradox.",
    "Tom is taller than 90% of his class. Can you conclude his exact percentile rank?",
    "If I flip a fair coin 10 times and get 10 heads, what is the probability the next flip is heads?",
    "Three switches, one lamp upstairs; you may go up once. How do you identify the switch?",
    "Explain Occam's razor and when it misleads.",
    "A train leaves at 60 km/h; a car leaves 1 hour later at 90 km/h. When does it catch up?",
    "Why is arguing from a small sample unreliable? Give a concrete example.",
    "Sort these by logical dependency: build foundation, frame walls, inspect permits, paint walls.",
    "If everyone reduces consumption by 10%, does total consumption drop by 10%? Explain rebound effects.",
    "Two doors, one truth-teller and one liar; ask one question to find the exit.",
    "A cube is painted and cut into 27 unit cubes. How many small cubes have exactly two painted faces?",
    "Why can't you conclude causation from an A/B test with a confounder? Example.",
    "Explain base rates with a rare disease testing puzzle.",
]


### `general_knowledge`

In [15]:
# general_knowledge: 25 canonical prompts
CATEGORIES['general_knowledge'] = [
    "What is the capital of Australia?",
    "Describe the causes of World War I.",
    "Explain photosynthesis in simple terms.",
    "Who wrote Pride and Prejudice?",
    "What is the boiling point of water at sea level?",
    "Name the planets of the solar system in order from the sun.",
    "What is the difference between weather and climate?",
    "Explain what the Internet's DNS system does.",
    "When did the Berlin Wall fall, and what followed?",
    "What are the three states of matter and their phase transitions?",
    "Who was Ada Lovelace and why is she significant in computing?",
    "Explain why the sky is blue during the day but red at sunset.",
    "What does the immune system do when it encounters a virus for the first time?",
    "List the branches of the US government and their primary functions.",
    "What causes tides on Earth?",
    "Explain the difference between AC and DC current.",
    "What is the significance of the Rosetta Stone?",
    "How do vaccines produce immunity?",
    "What is the largest ocean and roughly how much of Earth's surface does it cover?",
    "Explain how a refrigerator moves heat.",
    "What is compound interest and why does it grow exponentially?",
    "Describe the water cycle in four steps.",
    "Why do seasons occur, and why are they opposite in the two hemispheres?",
    "What is GDP and what does it fail to measure?",
    "Explain how noise-cancelling headphones work.",
]


### `writing`

In [16]:
# writing: 25 canonical prompts
CATEGORIES['writing'] = [
    "Write a haiku about autumn rain.",
    "Draft a polite email declining a meeting invitation.",
    "Compose a short poem about the ocean.",
    "Write a product description for a wooden coffee mug.",
    "Summarize the plot of Hamlet in three sentences.",
    "Rewrite this sentence to be clearer: 'The utilization of the aforementioned methodology was undertaken by the team.'",
    "Write a short story opening set in a lighthouse during a storm.",
    "Draft a release note announcing a breaking API change.",
    "Explain recursion to a 10-year-old in one paragraph.",
    "Write a witty one-line slogan for a keyboard-cleaning startup.",
    "Compose an apology letter for shipping a broken update, keeping customer trust.",
    "Describe a medieval market square using all five senses.",
    "Write a LinkedIn post about lessons from your first engineering failure.",
    "Turn these notes into a professional agenda: budget, hiring, roadmap, AOB.",
    "Write dialogue between a skeptical detective and an over-eager witness.",
    "Draft a README intro for a small open-source CLI tool.",
    "Write a toast celebrating a retiring codebase.",
    "Condense a 3-paragraph policy into 2 bullet points and 1 action item.",
    "Write a nursery rhyme that secretly teaches binary counting.",
    "Craft a rejection email that is kind but leaves no ambiguity.",
    "Write a user-facing error message for a failed payment, with next steps.",
    "Draft a changelog entry users will actually read, for a speed improvement.",
    "Write a 3-tweet thread explaining database indexes for founders.",
    "Compose a conference talk abstract on caching, under 120 words.",
    "Rewrite legalistic terms into plain English: a data retention clause.",
]


### `long_context`

In [17]:
# long_context: 18 canonical prompts
CATEGORIES['long_context'] = [
    "Summarize the following multi-paragraph document in three bullet points: Chapter 1 covers data pipelines and their failure modes, including schema drift, late-arriving events, and partition skew. Chapter 2 covers monitoring, defining SLOs for freshness and completeness, and alerting strategy. Chapter 3 covers backfills and idempotent job design.",
    "Given the meeting transcript about migrating services to a message queue, list every decision made and its stated rationale.",
    "From the API reference below, write the exact call to fetch a paginated list of datasets with an async client and retries enabled.",
    "Here is a rental agreement; state the notice period and the deposit return conditions in plain English.",
    "Read this changelog and tell me which version introduced the memory regression fix and what workaround existed before it.",
    "Given a protein FASTA header block, extract the organism, accession, and translation table in JSON.",
    "From this multi-email thread, identify the single decision owner and the current blocker.",
    "This contract clause is ambiguous; explain both readings and which favors the subcontractor.",
    "Summarize the following stack trace and suggested fix in two sentences.",
    "Given a 40-row CSV snippet of server logs, name the hour bucket with the most 5xx responses.",
    "Here are three product requirement drafts; list contradictions between them and propose a resolution.",
    "From this lecture transcript excerpt, write 4 flashcards with question and answer.",
    "This design doc proposes event sourcing; list operational risks it creates.",
    "From the error budget policy below, decide if a risky deploy is allowed today.",
    "Given a cron spec and holidays list, find dates the job should skip.",
    "From two candidate architectures below, build a comparison table with costs.",
    "Extract all API breaking changes from these 6 release notes into a table.",
    "Given a log excerpt spanning a failover, reconstruct the incident timeline.",
]


### `instruction_following`

In [18]:
# instruction_following: 20 canonical prompts
CATEGORIES['instruction_following'] = [
    "Rewrite this sentence in exactly five words: the system failed unexpectedly today.",
    "Answer with only the JSON key names, comma separated, no prose: {\"user\": {\"name\": \"Ada\", \"roles\": [\"admin\"]}}",
    "List three steps for rotating an API key. Use numbered markdown list format.",
    "Respond with a single word: what HTTP status means 'not found'?",
    "Translate 'deploy on Friday afternoon' into three different tones: formal, casual, humorous.",
    "Give the answer as a Python dict with keys a and b: 7+3 and 7*3.",
    "Write a commit message in conventional-commit style for adding a retry to the auth client.",
    "Produce exactly three bullet points about cache invalidation, each under 12 words.",
    "Output only the second paragraph of your answer, in all caps.",
    "End every sentence of your reply with the word 'confirmed'.",
    "Convert this list to a YAML array: apples, bananas, cherries.",
    "Answer with a table: column names lang, year; rows Python 1991, Rust 2010.",
    "Begin your response with 'Step 1:' and use exactly two steps total.",
    "Rewrite the sentence without any form of the verb 'to be': 'The cache is refreshed by the worker.'",
    "Give a one-sentence answer containing the number 42 and the word 'namespace'.",
    "Respond in JSON: {\"ok\": true} with an extra field for the current topic.",
    "Write the word 'banana' backwards, then give its length as a number.",
    "List exactly four colors, sorted alphabetically, comma-separated, nothing else.",
    "Answer in two sentences max: what makes a good code review comment?",
    "Respond with a haiku (5-7-5) about late-night debugging, no preamble.",
]


In [19]:
# Add deterministic arithmetic coverage after the full canonical corpus.
def arithmetic(n=120, seed=73):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        a, b, op = rng.randint(12, 99), rng.randint(12, 99), rng.choice(['+', '-', '*'])
        if op == '*' and max(a, b) > 30:
            a, b = min(a, 30), min(b, 30)
        forms = [
            f'Compute exactly: {a} {op} {b}.',
            f'What is {a} {op} {b}? Show the result only.',
            f'Calculate {a} {op} {b} and explain the carries/borrows.',
            f'Answer as JSON: {{"result": <value>}} for {a} {op} {b}.',
        ]
        out.append(rng.choice(forms))
    return out

def prompt_bank():
    rows = [(category, prompt) for category, prompts in CATEGORIES.items() for prompt in prompts]
    rows.extend(('math_generated', prompt) for prompt in arithmetic(ARITHMETIC_PROMPTS))
    return list(dict.fromkeys(rows))

BANK = prompt_bank()
assert sum(map(len, CATEGORIES.values())) == 498
assert len(BANK) >= 498
print({'curated_prompts': 498, 'generated_arithmetic_draws': ARITHMETIC_PROMPTS, 'unique_full_bank': len(BANK)})


{'curated_prompts': 498, 'generated_arithmetic_draws': 120, 'unique_full_bank': 594}


In [20]:
# Self-contained model, hook, metric, cache, clustering, and router utilities.
# No imports below refer to this repository.
def stage(root, name, run_id):
    d = root / 'results' / name
    for sub in ('csv', 'json', 'plots', 'logs', 'traces'): (d / sub).mkdir(parents=True, exist_ok=True)
    def log(event, **kw):
        with (d / 'logs' / 'events.jsonl').open('a', encoding='utf-8') as f:
            f.write(json.dumps({'time_utc': datetime.now(timezone.utc).isoformat(), 'run_id': run_id, 'stage': name, 'event': event, **kw}, default=str) + '\n')
    log('started'); return d, log

def save_csv(path, rows):
    if not rows: return
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def save_json(path, value): path.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')

def plot(path, fn):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4.8))
    try: fn(ax); fig.tight_layout(); fig.savefig(path, dpi=160)
    finally: plt.close(fig)

def device_of(model): return model.device if hasattr(model, 'device') else next(model.parameters()).device
def resident_layer(model, fallback):
    # device_map='auto' leaves offloaded layers as meta tensors between forwards.
    mapped = getattr(model, 'hf_device_map', {})
    gpu = sorted(int(k.rsplit('.', 1)[-1]) for k, v in mapped.items() if k.startswith('model.layers.') and isinstance(v, int))
    if gpu: return f'model.layers.{gpu[len(gpu)//2]}'
    for layer in layer_names(model):
        if all(p.device.type != 'meta' for p in get_mlp(model, layer).parameters()): return layer
    return fallback
def layer_names(model): return [f'model.layers.{i}' for i in range(model.config.num_hidden_layers)]
def planned_layers(model, n):
    total = model.config.num_hidden_layers
    ids = sorted(dict.fromkeys(np.linspace(max(1, total//10), min(total-2, total-1-total//10), min(n, total-2)).round().astype(int)))
    return [f'model.layers.{i}' for i in ids]
def get_mlp(model, layer): return model.get_submodule(f'{layer}.mlp')
def partition(inter, size): return [list(range(i, min(i + size, inter))) for i in range(0, inter, size)]
def down_weight(model, layer):
    # Resolve Accelerate CPU/disk-offloaded parameters as well as resident weights.
    down = get_mlp(model, layer).down_proj; p = down.weight
    if p.device.type != 'meta': return p.detach()
    candidates = [(down, ('weight',)), (get_mlp(model, layer), ('down_proj.weight', 'weight')),
                  (model.get_submodule(layer), ('mlp.down_proj.weight', 'down_proj.weight'))]
    for module, keys in candidates:
        mapping = getattr(getattr(module, '_hf_hook', None), 'weights_map', None)
        if mapping is None: continue
        for key in keys:
            try:
                if key in mapping: return mapping[key].detach()
            except Exception:
                try: return mapping[key].detach()
                except Exception: pass
    raise RuntimeError(f'Could not materialize offloaded down_proj weight for {layer}')
def z_importance(z, groups): return torch.stack([(z[..., g].float() ** 2).mean(-1) for g in groups], -1)
def output_importance(z, weight, groups):
    wt = weight.float().T.to(z.device)
    return torch.stack([(z[..., g].float() @ wt[g]).norm(dim=-1) for g in groups], -1)
def mask_top(groups, ordering, inter, fraction, device):
    keep = torch.zeros(inter, dtype=torch.bool, device=device); k = max(1, round(fraction * len(groups)))
    ids = torch.cat([torch.tensor(groups[int(e)], device=device) for e in ordering[:k]])
    keep[ids.long()] = True; return keep
def metrics(a, b):
    a, b = a.float(), b.float(); pa, pb = torch.softmax(a, -1), torch.softmax(b, -1)
    return {'cosine': float(torch.nn.functional.cosine_similarity(a.flatten(), b.flatten(), dim=0)), 'top1_match': int(a.argmax() == b.argmax()), 'top10_overlap': len(set(a.topk(10).indices.tolist()) & set(b.topk(10).indices.tolist()))/10, 'kl': float(torch.nn.functional.kl_div(pa.clamp_min(1e-12).log(), pb, reduction='batchmean'))}

class CaptureZ:
    def __init__(self): self.handles=[]; self.data={}
    def add(self, model, layer):
        key=f'{layer}.down_proj'
        self.handles.append(model.get_submodule(f'{layer}.mlp.down_proj').register_forward_hook(lambda m, inp, out, key=key: self.data.__setitem__(key, inp[0].detach())))
    def close(self):
        for h in self.handles: h.remove()
        self.handles=[]
class MaskDown:
    def __init__(self): self.handles=[]; self.mask=None; self.position=None; self.batch_masks=None; self.batch_positions=None
    def add(self, model, layer):
        def hook(mod, args, kwargs):
            z=args[0] if args else kwargs['hidden_states']
            if self.batch_masks is not None:
                z=z.clone()
                for row, position in enumerate(self.batch_positions): z[row, position] *= self.batch_masks[row].to(z.device, z.dtype)
            elif self.mask is not None:
                z=z.clone(); z[:, self.position] *= self.mask.to(z.device, z.dtype)
            return ((z,)+args[1:], kwargs) if args else (args, {**kwargs, 'hidden_states': z})
        self.handles.append(model.get_submodule(f'{layer}.mlp.down_proj').register_forward_pre_hook(hook, with_kwargs=True))
    def set(self, mask, position): self.mask, self.position, self.batch_masks, self.batch_positions = mask, position, None, None
    def set_batch(self, masks, positions): self.batch_masks, self.batch_positions, self.mask, self.position = masks, list(positions), None, None
    def close(self):
        for h in self.handles: h.remove()
        self.handles=[]

def capture_prompt(model, tokenizer, prompt, layers, groups):
    zs, xs, handles = CaptureZ(), {}, []
    for layer in layers:
        def xhook(mod, args, kwargs, key=layer): xs[key] = (args[0] if args else kwargs['hidden_states']).detach()
        handles.append(get_mlp(model, layer).register_forward_pre_hook(xhook, with_kwargs=True)); zs.add(model, layer)
    try:
        enc=tokenizer(prompt, return_tensors='pt').to(device_of(model))
        with torch.no_grad(): model(**enc)
        out={}
        for layer in layers:
            z=zs.data[f'{layer}.down_proj'][0,1:].float(); w=down_weight(model, layer).to(z.device)
            out[layer]={'x': xs[layer][0,1:].float().cpu().numpy().astype(np.float16), 'z': z.cpu().numpy().astype(np.float16), 'imp': output_importance(z, w, groups).cpu().numpy().astype(np.float32)}
        return out
    finally:
        zs.close()
        for h in handles: h.remove()

def cache_file(cache, layer, index, prompt): return cache / f'{layer.replace(".", "_")}__{index:04d}_{hashlib.sha256(prompt.encode()).hexdigest()[:16]}.npz'
def load_layer(cache, layer):
    xs=[]; zs=[]; imps=[]; pids=[]; cats=[]
    for i,(cat,prompt) in enumerate(BANK):
        d=np.load(cache_file(cache, layer, i, prompt)); n=len(d['x'])
        xs.append(d['x'].astype(np.float32)); zs.append(d['z'].astype(np.float32)); imps.append(d['imp'].astype(np.float32)); pids.append(np.full(n,i)); cats += [cat]*n
    return np.concatenate(xs), np.concatenate(zs), np.concatenate(imps), np.concatenate(pids), np.asarray(cats)

def kmeans(x, k, seed=73, iterations=40):
    rng=np.random.default_rng(seed); x=np.asarray(x, np.float32); k=min(max(1,k),len(x)); centers=x[rng.choice(len(x),k,replace=False)].copy()
    for _ in range(iterations):
        sample=x[rng.choice(len(x),min(len(x),4096),replace=False)]; labels=torch.cdist(torch.from_numpy(sample),torch.from_numpy(centers)).argmin(1).numpy()
        for c in range(k):
            if (labels==c).any(): centers[c]=sample[labels==c].mean(0)
    return torch.cdist(torch.from_numpy(x),torch.from_numpy(centers)).argmin(1).numpy()
def cluster_intra_corr(z, assignment, seed=73):
    signatures=np.asarray(z,np.float32).T; sizes=np.bincount(assignment); sums=np.zeros((len(sizes),signatures.shape[1]),np.float64); np.add.at(sums,assignment,signatures); means=sums/np.maximum(sizes[:,None],1)
    rng=np.random.default_rng(seed); values=[]
    for i in rng.permutation(len(assignment))[:min(512,len(assignment))]:
        if sizes[assignment[i]] > 1:
            a=signatures[i]-signatures[i].mean(); b=means[assignment[i]]-means[assignment[i]].mean(); denom=np.linalg.norm(a)*np.linalg.norm(b)
            if denom: values.append(float(a@b/denom))
    return float(np.mean(values)) if values else float('nan')
def mutual_information(a, b, n_a=None, n_b=None):
    n_a=int(a.max())+1 if n_a is None else n_a; n_b=int(b.max())+1 if n_b is None else n_b; joint=np.zeros((n_a,n_b)); np.add.at(joint,(a,b),1); joint/=joint.sum(); pa,pb=joint.sum(1,keepdims=True),joint.sum(0,keepdims=True); valid=joint>0
    return float((joint[valid]*np.log2(joint[valid]/(pa@pb)[valid])).sum())
def pr(pred, truth): return float(np.mean([len(set(a)&set(b))/len(a) for a,b in zip(pred,truth)]))
def centroid_scores(xtr, itr, xho, k):
    top=np.argsort(-itr,axis=1)[:,:k]; xn=xtr/np.maximum(np.linalg.norm(xtr,axis=1,keepdims=True),1e-8); centers=np.zeros((itr.shape[1],xtr.shape[1]),np.float32)
    for e in range(len(centers)):
        m=(top==e).any(1)
        if m.any(): centers[e]=xn[m].mean(0)
    centers/=np.maximum(np.linalg.norm(centers,axis=1,keepdims=True),1e-8); return (xho/np.maximum(np.linalg.norm(xho,axis=1,keepdims=True),1e-8))@centers.T
def linear_scores(xtr, itr, xho):
    # CPU avoids competing with the loaded model for Kaggle GPU memory.
    x=torch.from_numpy(xtr).float(); y=torch.from_numpy(np.log1p(itr)).float(); mu=x.mean(0,keepdim=True); sd=x.std(0,keepdim=True).clamp_min(1e-5); x=(x-mu)/sd
    m=torch.nn.Linear(x.shape[1],y.shape[1]); opt=torch.optim.AdamW(m.parameters(),lr=.02,weight_decay=1e-4)
    for _ in range(PROBE_EPOCHS): opt.zero_grad(); torch.nn.functional.smooth_l1_loss(m(x),y).backward(); opt.step()
    with torch.no_grad(): return m((torch.from_numpy(xho).float()-mu)/sd).numpy()
def knn_scores(xtr, itr, xho, neighbors=16, batch=128):
    # Exact cosine kNN, batched so the full holdout × train matrix is never materialized.
    train=xtr/np.maximum(np.linalg.norm(xtr,axis=1,keepdims=True),1e-8); hold=xho/np.maximum(np.linalg.norm(xho,axis=1,keepdims=True),1e-8); out=[]; neighbors=min(neighbors,len(train))
    for start in range(0,len(hold),batch):
        similarity=hold[start:start+batch]@train.T; idx=np.argpartition(-similarity,neighbors-1,axis=1)[:,:neighbors]; out.append(itr[idx].mean(1))
    return np.concatenate(out)
def router_predictions(xtr, itr, xho, zho, granularity, k, rng):
    experts=itr.shape[1]; freq=np.bincount(np.argsort(-itr,axis=1)[:,:k].ravel(),minlength=experts); scores={'centroid':centroid_scores(xtr,itr,xho,k),'knn':knn_scores(xtr,itr,xho),'linear_probe':linear_scores(xtr,itr,xho),'z_energy_posthoc':-(zho.reshape(len(zho),experts,granularity)**2).mean(2)}
    answer={name:np.argsort(-value,axis=1)[:,:k] for name,value in scores.items()}; answer['frequency']=np.tile(np.argsort(-freq)[:k],(len(xho),1)); answer['random']=np.stack([rng.permutation(experts)[:k] for _ in range(len(xho))]); return answer
def plot_m4_summary(path, rows):
    keys=sorted(set((r['k'],r['router']) for r in rows)); labels=[f'{name}@{k}' for k,name in keys]; values=[np.mean([r['holdout_recall'] for r in rows if (r['k'],r['router'])==key]) for key in keys]
    plot(path,lambda ax:(ax.bar(labels,values),ax.tick_params(axis='x',rotation=55,labelsize=8),ax.set(ylabel='mean holdout recall',title='M4 router recall by K'),ax.grid(axis='y',alpha=.3)))
def plot_m4b_summary(path, rows):
    plot(path,lambda ax:([ax.plot(sorted(set(r['granularity'] for r in rows if r['router']==name)),[np.mean([r['holdout_recall'] for r in rows if r['router']==name and r['granularity']==g]) for g in sorted(set(r['granularity'] for r in rows if r['router']==name))],marker='o',label=name) for name in sorted(set(r['router'] for r in rows))],ax.legend(fontsize=8),ax.set(xlabel='neurons/expert',ylabel='mean holdout recall',title='M4b granularity'),ax.grid(alpha=.3)))


In [21]:
def run_one(model_key, model_id):
    run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); bundle=OUTPUT_ROOT/model_key; bundle.mkdir(exist_ok=True)
    meta,mlog=stage(bundle,'run_metadata',run_id)
    save_json(meta/'json'/'run_config.json', {'run_id':run_id,'model_id':model_id,'prompts':len(BANK),'curated':sum(map(len,CATEGORIES.values())),'arithmetic':ARITHMETIC_PROMPTS,'settings':{'group_size':GROUP_SIZE,'m1_layers':M1_LAYERS,'router_layers':ROUTER_LAYERS}})
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    tokenizer=AutoTokenizer.from_pretrained(model_id)
    model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype,device_map='auto',low_cpu_mem_usage=True); model.eval()
    inter=int(model.config.intermediate_size); hidden=int(model.config.hidden_size); all_layers=layer_names(model); m1layers=planned_layers(model,M1_LAYERS); routerlayers=planned_layers(model,ROUTER_LAYERS); resident=resident_layer(model, routerlayers[len(routerlayers)//2]); routerlayers=sorted(set(routerlayers+[resident]), key=lambda x:int(x.rsplit('.',1)[-1])); groups=partition(inter,GROUP_SIZE)
    mlog('model_loaded',layers=len(all_layers),hidden=hidden,intermediate=inter,m1_layers=m1layers,router_layers=routerlayers)

    # M0 exact decomposition and permutation equivalence on a materialized FFN.
    d,log=stage(bundle,'m0_exactness',run_id); mlp=get_mlp(model,resident); resident_param=next(mlp.parameters()); probe=torch.randn(2,3,hidden,device=resident_param.device,dtype=resident_param.dtype); m0=[]
    for size in [s for s in (64,128,256,512) if s<=inter]:
        gs=partition(inter,size); z=torch.nn.functional.silu(mlp.gate_proj(probe))*mlp.up_proj(probe); dense=mlp.down_proj(z); summed=sum((z[...,g]@mlp.down_proj.weight[:,g].T) for g in gs); perm=torch.randperm(inter,device=z.device); zg=z[...,perm]; wp=mlp.down_proj.weight[:,perm]; reordered=torch.nn.functional.linear(zg,wp)
        m0.append({'group_size':size,'n_experts':len(gs),'rel_err_sum':float((dense-summed).norm()/(dense.norm()+1e-12)),'rel_err_permutation':float((dense-reordered).norm()/(dense.norm()+1e-12))})
    # Full-model all-experts control: the masking mechanism itself must preserve logits at 100% retention.
    control_prompt=BANK[0][1]; control_enc=tokenizer(control_prompt,return_tensors='pt').to(device_of(model))
    with torch.no_grad(): control_dense=model(**control_enc).logits[0,-1].float().cpu()
    control_mask=MaskDown(); control_mask.add(model,resident); control_mask.set(torch.ones(inter,dtype=torch.bool),control_enc['input_ids'].shape[1]-1)
    try:
        with torch.no_grad(): control_masked=model(**control_enc).logits[0,-1].float().cpu()
    finally: control_mask.close()
    full_forward=metrics(control_masked,control_dense); passed=all(r['rel_err_sum']<1e-4 and r['rel_err_permutation']<1e-4 for r in m0) and full_forward['cosine']>.9999 and full_forward['top1_match']==1
    save_csv(d/'csv'/'exactness_report.csv',m0); save_json(d/'json'/'m0_verdict.json',{'passed':passed,'resident_layer':resident,'full_model_all_experts':full_forward,'rows':m0}); plot(d/'plots'/'relative_error.png',lambda ax:(ax.semilogy([r['group_size'] for r in m0],[max(r['rel_err_sum'],1e-16) for r in m0],marker='o',label='sum'),ax.semilogy([r['group_size'] for r in m0],[max(r['rel_err_permutation'],1e-16) for r in m0],marker='s',label='permutation'),ax.legend(),ax.set(xlabel='neurons/expert',ylabel='relative error',title='M0 exactness'),ax.grid(alpha=.3))); log('completed',passed=passed,full_model_cosine=full_forward['cosine'])
    if not passed: raise RuntimeError('M0 failed; stopping model study')

    # M1 full-bank multi-token oracle.  The dense FFN runs first; only down projection is masked.
    d,log=stage(bundle,'m1_oracle_sparsity',run_id); rows=[]
    for layer in m1layers:
        for pi,(cat,prompt) in enumerate(BANK,1):
            cap=CaptureZ(); cap.add(model,layer); enc=tokenizer(prompt,return_tensors='pt').to(device_of(model))
            with torch.no_grad(): base=model(**enc).logits[0].float().cpu()
            zall=cap.data[f'{layer}.down_proj'][0]; cap.close(); positions=np.unique(np.linspace(1,max(1,len(zall)-1),min(M1_POSITIONS,max(1,len(zall)-1))).round().astype(int)); w=down_weight(model,layer).to(zall.device); ranks=[torch.argsort(output_importance(zall[t:t+1],w,groups)[0],descending=True) for t in positions]; batch={'input_ids':enc['input_ids'].expand(len(positions),-1).contiguous(),'attention_mask':enc['attention_mask'].expand(len(positions),-1).contiguous()}; mask=MaskDown(); mask.add(model,layer)
            try:
                for frac in RETAIN:
                    keeps=torch.stack([mask_top(groups,order,inter,frac,zall.device) for order in ranks]); mask.set_batch(keeps,positions)
                    with torch.no_grad(): sparse_batch=model(**batch).logits.float().cpu()
                    for row,t in enumerate(positions): rows.append({'category':cat,'prompt_id':pi,'layer':layer,'position':int(t),'retain_fraction':frac,**metrics(sparse_batch[row,int(t)],base[int(t)])})
            finally: mask.close()
            if pi%10==0:
                save_csv(d/'csv'/'oracle_ffn_checkpoint.csv',rows)
                log('progress',layer=layer,prompts=pi,total=len(BANK),rows=len(rows))
    save_csv(d/'csv'/'oracle_ffn.csv',rows); summary=[{'retain_fraction':f,'cosine_mean':float(np.mean([r['cosine'] for r in rows if r['retain_fraction']==f])),'top1_mean':float(np.mean([r['top1_match'] for r in rows if r['retain_fraction']==f]))} for f in RETAIN]; save_csv(d/'csv'/'m1_summary.csv',summary); save_json(d/'json'/'m1_verdict.json',{'layers':m1layers,'prompts':len(BANK),'summary':summary}); plot(d/'plots'/'quality_vs_retention.png',lambda ax:(ax.plot([r['retain_fraction'] for r in summary],[r['cosine_mean'] for r in summary],marker='o'),ax.set(xlabel='retained experts',ylabel='mean logit cosine',title='M1 oracle sparsity'),ax.grid(alpha=.3))); log('completed',rows=len(rows))

    # Shared full-bank capture for M2/M3/M3b/M4/M4b.  Resumable per prompt/layer.
    d,log=stage(bundle,'m4_token_router',run_id); cache=bundle/'results'/'traces'/'full_bank_capture'; cache.mkdir(parents=True,exist_ok=True)
    for i,(_cat,prompt) in enumerate(BANK):
        missing=[layer for layer in routerlayers if OVERWRITE_CAPTURE or not cache_file(cache,layer,i,prompt).exists()]
        if missing:
            captured=capture_prompt(model,tokenizer,prompt,routerlayers,groups)
            for layer in missing: np.savez_compressed(cache_file(cache,layer,i,prompt),**captured[layer])
        if (i+1)%25==0: log('capture_progress',prompts=i+1,total=len(BANK))
    save_json(d/'json'/'capture_manifest.json',{'full_prompt_bank':len(BANK),'layers':routerlayers,'cache':str(cache),'fields':['x','z','imp']})

    # M2 task-specialization and M3 neuron clustering over every captured token.
    d2,l2=stage(bundle,'m2_task_specialization',run_id); d3,l3=stage(bundle,'m3_neuron_clustering',run_id); m2=[]; m3=[]
    for layer in routerlayers:
        x,z,imp,pid,cats=load_layer(cache,layer); tasks=sorted(set(cats)); energies=np.stack([(z[:,g]**2).mean(1) for g in groups],1); threshold=np.quantile(energies,.5,axis=0); active=energies>threshold
        probs=np.stack([active[cats==task].mean(0) for task in tasks]); task_dist=probs/np.maximum(probs.sum(0,keepdims=True),1e-12); entropy=-(np.clip(task_dist,1e-12,1)*np.log2(np.clip(task_dist,1e-12,1))).sum(0)/np.log2(len(tasks))
        for ti,task in enumerate(tasks):
            for e,p in enumerate(probs[ti]): m2.append({'layer':layer,'task':task,'expert_id':e,'p_active':float(p),'task_entropy':float(entropy[e])})
        assign=kmeans(z.T,len(groups)); sizes=np.bincount(assign,minlength=len(groups)); np.save(d3/'traces'/f'{layer.replace(".","_")}_assignments.npy',assign); reorder_err=None
        if layer == resident:
            test_mlp=get_mlp(model,layer); test_param=next(test_mlp.parameters()); order=torch.from_numpy(np.argsort(assign)).to(test_param.device); testx=torch.randn(1,2,hidden,device=test_param.device,dtype=test_param.dtype); dense=test_mlp.down_proj(torch.nn.functional.silu(test_mlp.gate_proj(testx))*test_mlp.up_proj(testx)); zp=torch.nn.functional.silu(torch.nn.functional.linear(testx,test_mlp.gate_proj.weight[order]))*torch.nn.functional.linear(testx,test_mlp.up_proj.weight[order]); permuted=torch.nn.functional.linear(zp,test_mlp.down_proj.weight[:,order]); reorder_err=float((dense-permuted).norm()/(dense.norm()+1e-12))
        intra_corr=cluster_intra_corr(z,assign); m3.append({'layer':layer,'n_clusters':int((sizes>0).sum()),'cluster_size_mean':float(sizes[sizes>0].mean()),'cluster_size_std':float(sizes[sizes>0].std()),'intra_corr_mean':intra_corr,'cluster_reorder_rel_err':reorder_err}); save_csv(d2/'csv'/'expert_task_statistics_checkpoint.csv',m2); save_csv(d3/'csv'/'cluster_statistics_checkpoint.csv',m3); l2('layer_completed',layer=layer,tokens=len(z),intra_corr_mean=intra_corr,cluster_reorder_rel_err=reorder_err); del x,z,imp
    save_csv(d2/'csv'/'expert_task_statistics.csv',m2); save_json(d2/'json'/'task_specialization.json',{'prompts':len(BANK),'rows':m2}); save_csv(d3/'csv'/'cluster_statistics.csv',m3); save_json(d3/'json'/'cluster_specialization.json',{'prompts':len(BANK),'rows':m3}); plot(d2/'plots'/'task_entropy.png',lambda ax:(ax.hist([r['task_entropy'] for r in m2],bins=30),ax.set(xlabel='normalized task entropy',ylabel='experts',title='M2 specialization'))); plot(d3/'plots'/'cluster_sizes.png',lambda ax:(ax.bar([r['layer'] for r in m3],[r['cluster_size_mean'] for r in m3]),ax.tick_params(axis='x',rotation=25),ax.set(ylabel='mean cluster size',title='M3 clustering'))); l2('completed',rows=len(m2)); l3('completed',rows=len(m3))

    # M3b: finer neuron clusters and token-space routing-fingerprint information.
    d,l=stage(bundle,'m3b_granularity_option2',run_id); m3b=[]
    for layer in routerlayers:
        _x,z,imp,_pid,cats=load_layer(cache,layer); tasks=sorted(set(cats)); taskids=np.array([tasks.index(c) for c in cats]); basek=max(1,inter//GROUP_SIZE)
        for k in sorted(set([basek,min(inter,basek*2),min(inter,basek*4)])):
            ass=kmeans(z.T,k); g=[np.where(ass==i)[0].tolist() for i in range(k) if (ass==i).any()]; eng=np.stack([(z[:,q]**2).mean(1) for q in g],1); p=np.stack([(eng[cats==t]>np.quantile(eng,.5,axis=0)).mean(0) for t in tasks]); task_dist=p/np.maximum(p.sum(0,keepdims=True),1e-12); ent=-(np.clip(task_dist,1e-12,1)*np.log2(np.clip(task_dist,1e-12,1))).sum(0)/np.log2(len(tasks)); tc=kmeans(imp/np.maximum(np.linalg.norm(imp,axis=1,keepdims=True),1e-8),len(tasks)); mi=mutual_information(tc,taskids,len(tasks),len(tasks)); perm_rng=np.random.default_rng(73); null=np.array([mutual_information(tc,perm_rng.permutation(taskids),len(tasks),len(tasks)) for _ in range(200)]); p_value=float((null>=mi).mean()); m3b.append({'layer':layer,'clusters':k,'mean_task_entropy':float(ent.mean()),'fraction_entropy_lt_075':float((ent<.75).mean()),'token_task_mi_bits':mi,'token_task_mi_permutation_p_value':p_value,'mi_permutations':200}); save_csv(d/'csv'/'granularity_and_token_mi_checkpoint.csv',m3b); l('progress',layer=layer,clusters=k,rows=len(m3b))
    save_csv(d/'csv'/'granularity_and_token_mi.csv',m3b); save_json(d/'json'/'option2_verdict.json',{'prompts':len(BANK),'rows':m3b}); plot(d/'plots'/'m3b_summary.png',lambda ax:([ax.plot([r['clusters'] for r in m3b if r['layer']==layer],[r['mean_task_entropy'] for r in m3b if r['layer']==layer],marker='o',label=layer) for layer in routerlayers],ax.legend(),ax.set(xlabel='cluster count',ylabel='mean entropy',title='M3b granularity'),ax.grid(alpha=.3))); l('completed',rows=len(m3b))

    # M4 / M4b: prompt-disjoint split; input-only routers plus post-hoc z-energy ceiling.
    dm4,l4=stage(bundle,'m4_token_router',run_id); dm4b,l4b=stage(bundle,'m4b_granularity_router',run_id); rng=np.random.default_rng(73); train_ids=set(rng.permutation(len(BANK))[:round(.7*len(BANK))]); m4=[]; m4b=[]
    for layer in routerlayers:
        x,z,imp,pid,_cats=load_layer(cache,layer); tr=np.isin(pid,list(train_ids)); xtr,xho=x[tr],x[~tr]; ztr,zho=z[tr],z[~tr]
        for gran in [g for g in (128,256,512,1024) if g<=inter and inter%g==0]:
            gs=partition(inter,gran); w=down_weight(model,layer).cpu();
            def score(a): return np.concatenate([output_importance(torch.from_numpy(a[s:s+256]),w,gs).numpy() for s in range(0,len(a),256)])
            itr,iho=score(ztr),score(zho); e=itr.shape[1]; k=max(1,round(.25*e)); truth=np.argsort(-iho,axis=1)[:,:k]; choices=router_predictions(xtr,itr,xho,zho,gran,k,rng)
            for name,pred in choices.items():
                m4b.append({'layer':layer,'granularity':gran,'n_experts':e,'k':k,'router':name,'holdout_recall':pr(pred,truth)})
            if gran==GROUP_SIZE:
                for eval_k in sorted(set(min(e,k0) for k0 in (8,19,30))):
                    eval_truth=np.argsort(-iho,axis=1)[:,:eval_k]; eval_choices=router_predictions(xtr,itr,xho,zho,gran,eval_k,rng)
                    for name,pred in eval_choices.items(): m4.append({'layer':layer,'granularity':gran,'n_experts':e,'k':eval_k,'router':name,'holdout_recall':pr(pred,eval_truth)})
        save_csv(dm4/'csv'/'router_vs_oracle_checkpoint.csv',m4); save_csv(dm4b/'csv'/'router_granularity_checkpoint.csv',m4b); l4('layer_completed',layer=layer,rows=len(m4))
    save_csv(dm4/'csv'/'router_vs_oracle.csv',m4); save_json(dm4/'json'/'router_summary.json',{'prompt_split':{'train':len(train_ids),'holdout':len(BANK)-len(train_ids),'seed':73},'k_values':sorted(set(r['k'] for r in m4)),'rows':m4}); save_csv(dm4b/'csv'/'router_granularity.csv',m4b); save_json(dm4b/'json'/'granularity_router_verdict.json',{'full_prompt_bank':len(BANK),'rows':m4b}); plot_m4_summary(dm4/'plots'/'router_recall.png',m4); plot_m4b_summary(dm4b/'plots'/'granularity_recall.png',m4b); l4('completed',rows=len(m4)); l4b('completed',rows=len(m4b))

    # M4c real masked-forward quality for all prompts; checkpointed every ten prompts.
    d,l=stage(bundle,'m4c_zenergy',run_id); layer=routerlayers[len(routerlayers)//2]; m4c=[]
    for pi,(cat,prompt) in enumerate(BANK,1):
        cap=CaptureZ(); cap.add(model,layer); enc=tokenizer(prompt,return_tensors='pt').to(device_of(model))
        with torch.no_grad(): base=model(**enc).logits[0].float().cpu()
        zall=cap.data[f'{layer}.down_proj'][0]; cap.close(); w=down_weight(model,layer).to(zall.device); positions=np.unique(np.linspace(1,max(1,len(zall)-1),min(M4C_POSITIONS,max(1,len(zall)-1))).round().astype(int)); masker=MaskDown(); masker.add(model,layer)
        try:
            batch={'input_ids':enc['input_ids'].expand(len(positions),-1).contiguous(),'attention_mask':enc['attention_mask'].expand(len(positions),-1).contiguous()}; ranks={'oracle':[],'z_energy':[],'random':[]}
            for t in positions:
                z=zall[t]; ranks['oracle'].append(torch.argsort(output_importance(z[None],w,groups)[0],descending=True)); ranks['z_energy'].append(torch.argsort(z_importance(z[None],groups)[0],descending=True)); ranks['random'].append(torch.randperm(len(groups),device=z.device))
            for name,orders in ranks.items():
                for frac in M4C_RETAIN:
                    keeps=torch.stack([mask_top(groups,order,inter,frac,zall.device) for order in orders]); masker.set_batch(keeps,positions)
                    with torch.no_grad(): sparse_batch=model(**batch).logits.float().cpu()
                    for row,t in enumerate(positions): m4c.append({'category':cat,'prompt_id':pi,'layer':layer,'position':int(t),'ranker':name,'retain_fraction':frac,**metrics(sparse_batch[row,int(t)],base[int(t)])})
        finally: masker.close()
        if pi%10==0: save_csv(d/'csv'/'zenergy_quality_checkpoint.csv',m4c); l('progress',prompts=pi,total=len(BANK),rows=len(m4c))
    save_csv(d/'csv'/'zenergy_quality.csv',m4c); zs=[{'ranker':name,'retain_fraction':frac,'cosine_mean':float(np.mean([r['cosine'] for r in m4c if r['ranker']==name and r['retain_fraction']==frac]))} for name in ('oracle','z_energy','random') for frac in M4C_RETAIN]; save_csv(d/'csv'/'zenergy_summary.csv',zs)
    # M4c part B: measured dense versus gathered down-projection on a resident layer.
    wc=[]
    if torch.cuda.is_available():
        bench_layer=resident_layer(model,layer); bw=down_weight(model,bench_layer).to('cuda'); bz=torch.randn(32,inter,device='cuda',dtype=bw.dtype)
        def timed(fn):
            for _ in range(10): fn()
            torch.cuda.synchronize(); start=time.perf_counter()
            for _ in range(50): fn()
            torch.cuda.synchronize(); return (time.perf_counter()-start)*1000/50
        dense=timed(lambda: torch.nn.functional.linear(bz,bw)); wc.append({'mode':'dense','retain_fraction':1.0,'milliseconds':dense,'ratio_vs_dense':1.0})
        for frac in M4C_RETAIN:
            kept_neurons=min(inter,max(GROUP_SIZE,round(frac*len(groups))*GROUP_SIZE)); idx=torch.arange(kept_neurons,device='cuda'); gathered=timed(lambda: torch.nn.functional.linear(bz[:,idx],bw[:,idx])); wc.append({'mode':'gathered_regular','retain_fraction':frac,'milliseconds':gathered,'ratio_vs_dense':gathered/dense})
    else: wc.append({'mode':'skipped_no_cuda','retain_fraction':None,'milliseconds':None,'ratio_vs_dense':None})
    save_csv(d/'csv'/'downproj_wallclock.csv',wc); save_json(d/'json'/'zenergy_verdict.json',{'layer':layer,'prompts':len(BANK),'summary':zs,'wall_clock':wc}); plot(d/'plots'/'zenergy_quality.png',lambda ax:([ax.plot([r['retain_fraction'] for r in zs if r['ranker']==name],[r['cosine_mean'] for r in zs if r['ranker']==name],marker='o',label=name) for name in ('oracle','z_energy','random')],ax.legend(),ax.set(xlabel='retained experts',ylabel='mean logit cosine',title='M4c masked forward quality'),ax.grid(alpha=.3))); l('completed',rows=len(m4c),wall_clock_rows=len(wc)); mlog('completed')
    del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return {'model':model_id,'run_id':run_id,'bundle':str(bundle),'status':'completed'}


In [22]:
# Run models sequentially.  Results remain usable if a later model fails.
completed = []
for model_key in SELECTED_MODELS:
    print(f'\n===== BEGIN {model_key}: {MODELS[model_key]} =====')
    completed.append(run_one(model_key, MODELS[model_key]))
    print(f'===== COMPLETE {model_key} =====')
completed



===== BEGIN qwen3_4b: Qwen/Qwen3-4B =====


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
C:\Users\shubh\AppData\Local\Temp\ipykernel_4924\1659688775.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  m0.append({'group_size':size,'n_experts':len(gs),'rel_err_sum':float((dense-summed).norm()/(dense.norm()+1e-12)),'rel_err_permutation':float((dense-reordered).norm()/(dense.norm()+1e-12))})


RuntimeError: M0 failed; stopping model study